# 05 — Predictive Modeling: M5 Churn, Value, Profit, and PSM Support

**Owner:** M5 — Thư  
**Purpose:** explain and run the M5 modeling workflow, review model outputs, and generate the Propensity Score file requested by M6.

This notebook is still script-backed: core reusable logic lives in `scripts/modeling.py` and `scripts/psm_propensity_score.py`. The notebook is for review, explanation, and handoff, not for duplicating all production code.

## Notebook structure

1. Validate core input files.
2. Optionally run the full M5 pipeline.
3. Explain and review the main output tables:
   - churn model benchmark
   - calibration summary
   - active/value models
   - top-K ranking and profit scenario tables
   - M6 handoff files
4. Explain and optionally run the PSM propensity-score support step for M6.

**Important:** PSM support is separate from the core churn model. It does not replace the final churn champion; it only creates `propensity_score = P(is_treated = 1 | observed covariates)` for M6 matching.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists() and (PROJECT_ROOT.parent / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'config' / 'paths.yaml'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = MODELS_DIR / 'reports'
M6_HANDOFF_DIR = MODELS_DIR / 'm6_handoff'
DIAGNOSTICS_DIR = MODELS_DIR / 'diagnostics'
ARTIFACTS_DIR = MODELS_DIR / 'artifacts'
PSM_INPUTS_DIR = MODELS_DIR / 'psm_inputs'

print('Project root:', PROJECT_ROOT)
print('Config:', CONFIG_PATH)
print('Models:', MODELS_DIR)

Project root: C:\Users\Thuww\DDM-Churn-Project
Config: C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml
Models: C:\Users\Thuww\DDM-Churn-Project\models


## 1. Validate core inputs

This step checks whether the key repo inputs exist before running or reviewing M5.

M5's primary input is the **M4 household-level feature table**:

```text
models/final_ML_features.csv
```

This is treated as the feature handoff from M4. M5 does not rebuild M4 features here.

In [2]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(CONFIG_PATH)
input_checks

{
  "feature_table_csv": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv",
    "exists": true,
    "size_bytes": 693945
  },
  "transaction_master_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet",
    "exists": true,
    "size_bytes": 24336697
  },
  "customer_base_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet",
    "exists": true,
    "size_bytes": 122150
  }
}


{'feature_table_csv': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv',
  'exists': True,
  'size_bytes': 693945},
 'transaction_master_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet',
  'exists': True,
  'size_bytes': 24336697},
 'customer_base_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet',
  'exists': True,
  'size_bytes': 122150}}

## 2. Run the full M5 pipeline with `config/paths.yaml`

This is the notebook entry point for running the actual M5 script:

```bash
python scripts/modeling.py --config config/paths.yaml
```

The config file is:

```text
config/paths.yaml
```

The full pipeline trains/benchmarks churn models, calibrates the selected model, trains the value model, calculates expected profit, exports SHAP/diagnostics, and organizes M5 outputs by folder.

Because this can take time, `RUN_FULL_M5_PIPELINE` is set to `False` by default. Set it to `True` only when you want to regenerate all M5 outputs.


In [14]:
RUN_FULL_M5_PIPELINE = True

if RUN_FULL_M5_PIPELINE:
    import subprocess
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'modeling.py'),
        '--config',
        str(CONFIG_PATH),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

summary_path = PROJECT_ROOT / 'reports' / 'internal_briefs' / 'm5_pipeline_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Loaded pipeline summary from:', summary_path.relative_to(PROJECT_ROOT))
else:
    summary = {}
    print('No pipeline summary found yet. Set RUN_FULL_M5_PIPELINE=True to run M5.')

summary


Running: c:\Users\Thuww\DDM-Churn-Project\.venv\Scripts\python.exe C:\Users\Thuww\DDM-Churn-Project\scripts\modeling.py --config C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml
Loaded pipeline summary from: reports\internal_briefs\m5_pipeline_summary.json


{'version': 'v3_discounted_two_part_value_shap_seasonality',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'XGBoost weighted',
 'calibration_method': 'sigmoid',
 'champion_threshold': 0.13,
 'test_PR_AUC_calibrated': 0.3652138015042468,
 'test_F2_score_calibrated': 0.4583333333333333,
 'test_brier_score_calibrated': 0.09506485629007785,
 'active_model': 'Active Random Forest__isotonic',
 'value_model': 'Random Forest Regressor',
 'annual_discount_rate': 0.08,
 'base_profitable_customers': 0,
 'shap_status_file': 'C:\\Users\\Thuww\\DDM-Churn-Project\\reports\\internal_briefs\\M5_shap_status.json',
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models',
 'reports_outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\reports',
 'm6_handoff_outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\m6_handoff',
 'diagnostics_outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\

## 3. Churn model benchmark — what this table means

`model_metrics.csv` compares candidate churn classifiers before final business interpretation.

**Main purpose:** decide which model has useful churn-risk ranking signal.

Key metrics:

| Metric | Meaning |
|---|---|
| `PR-AUC` | Ranking quality for imbalanced churn data. More important than accuracy. |
| `ROC-AUC` | General class separation. Useful, but can look optimistic under imbalance. |
| `F2_score` | Classification score that weights recall more than precision. |
| `precision` | Among predicted churn-risk customers, how many actually churned. |
| `recall` | Among actual churners, how many the model caught. |
| `Brier score` | Probability quality; lower is better. |

Because churn is rare, global precision can look low. For M6, top-K precision and ranking lift are more important than global precision.

In [5]:
show_selected(
    model_metrics,
    [
        'model', 'tuned', 'features_used', 'best_cv_PR_AUC', 'best_cv_PR_AUC_std',
        'val_PR_AUC', 'val_ROC_AUC', 'val_F2_score', 'val_precision', 'val_recall',
        'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall', 'test_brier_score'
    ],
    n=10,
)

,model,tuned,features_used,best_cv_PR_AUC,best_cv_PR_AUC_std,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,test_PR_AUC,test_F2_score,test_precision,test_recall,test_brier_score
0,XGBoost weighted,True,numeric+categorical,0.368037,0.057377,0.377835,0.736790,0.515873,0.291045,0.639344,0.365214,0.459610,0.277311,0.550000,0.185405
1,Extra Trees balanced,True,numeric+categorical,0.355746,0.056799,0.373724,0.746817,0.486726,0.211538,0.721311,0.368512,0.472973,0.205882,0.700000,0.166141
2,Logistic Regression balanced,True,numeric+categorical,0.315523,0.044709,0.366964,0.747265,0.483559,0.183150,0.819672,0.341620,0.456432,0.181818,0.733333,0.205190
3,Random Forest balanced,True,numeric+categorical,0.366619,0.066833,0.308891,0.726950,0.488851,0.168142,0.934426,0.360901,0.456110,0.155425,0.883333,0.100048
4,Dummy prior,False,none,NaN,NaN,0.122000,0.500000,0.409946,0.122000,1.000000,0.120000,0.405405,0.120000,1.000000,0.105601


## 4. Calibration summary — why calibration matters

`calibration_summary.csv` compares raw, sigmoid, and isotonic probability outputs for the selected churn model.

**Why this matters:** expected profit uses churn probability directly. If raw model probability is inflated, expected profit will be misleading.

Typical interpretation:

- `raw_uncalibrated`: may preserve ranking but often has poor probability quality.
- `sigmoid`: smooth calibration, often keeps ranking resolution.
- `isotonic`: can improve Brier score but may create probability flat spots.

Final M5 should use the row where:

```text
selected_for_champion = True
```

This is the calibrated probability used as `p_churn_calibrated`.

In [6]:
show_selected(
    calibration_summary,
    [
        'champion_model', 'calibration_method', 'val_PR_AUC', 'val_ROC_AUC',
        'val_F2_score', 'val_precision', 'val_recall', 'val_brier_score',
        'val_threshold', 'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall',
        'test_brier_score', 'test_mean_predicted_probability', 'test_actual_positive_rate',
        'test_calibration_gap_mean_minus_actual', 'val_brier_gap_from_best',
        'calibration_selection_eligible', 'selected_for_champion'
    ],
    n=10,
)

,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,test_PR_AUC,test_F2_score,test_precision,test_recall,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,val_brier_gap_from_best,calibration_selection_eligible,selected_for_champion
0,XGBoost weighted,sigmoid,0.377835,0.73679,0.515873,0.291045,0.639344,0.096296,0.13,0.365214,0.458333,0.275000,0.55,0.095065,0.121438,0.12,0.001438,0.009176,True,True
1,XGBoost weighted,raw_uncalibrated,0.377835,0.73679,0.515873,0.291045,0.639344,0.186154,0.47,0.365214,0.459610,0.277311,0.55,0.185405,0.422645,0.12,0.302645,0.099033,False,False
2,XGBoost weighted,isotonic,0.369233,0.77305,0.517241,0.293233,0.639344,0.087120,0.09,0.307991,0.460894,0.279661,0.55,0.093761,0.126981,0.12,0.006981,0.000000,True,False


## 5. Active model and value model — what these tables mean

These two tables belong to the **value layer**, not the main churn classifier.

### 5.1 Active model

`active_model_metrics.csv` estimates:

```text
p_future_active = P(household generates revenue in the 60-day prediction window)
```

This is a supporting component. It should **not** be used as an independent targeting signal because most households may remain active, and the active target can overlap with the inverse of churn.

### 5.2 Value model

`value_model_metrics.csv` compares regressors for:

```text
log1p(discounted_future_revenue_60d | active)
```

The chosen value model estimates:

```text
predicted_discounted_value_60d_if_active
```

This is **short-term 60-day value**, not full lifetime CLV.

In [7]:
print('Active model metrics')
display(show_selected(
    active_model_metrics,
    [
        'active_model', 'calibration_method', 'val_PR_AUC', 'val_brier_score',
        'val_threshold', 'test_brier_score', 'test_mean_predicted_probability',
        'test_actual_positive_rate', 'test_calibration_gap_mean_minus_actual',
        'test_TN', 'test_FP', 'test_FN', 'test_TP'
    ],
    n=10,
))

print('Value model metrics')
display(show_selected(
    value_model_metrics,
    [
        'value_model', 'target', 'val_RMSE_log', 'val_MAE_log', 'val_R2_log',
        'val_MAE_revenue', 'test_RMSE_log', 'test_MAE_log', 'test_R2_log',
        'test_MAE_revenue'
    ],
    n=10,
))

Active model metrics


,active_model,calibration_method,val_PR_AUC,val_brier_score,val_threshold,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Random Forest,isotonic,0.984074,0.060805,0.01,0.070896,0.900935,0.892,0.008935,0,54,0,446
1,Active Logistic Regression,isotonic,0.983525,0.061413,0.01,0.075268,0.903431,0.892,0.011431,0,54,0,446
2,Active Random Forest,raw_uncalibrated,0.984240,0.064937,0.46,0.071005,0.888579,0.892,-0.003421,0,54,0,446
3,Active Logistic Regression,raw_uncalibrated,0.983875,0.068618,0.12,0.072582,0.891017,0.892,-0.000983,0,54,0,446
4,Active Random Forest,sigmoid,0.984240,0.069956,0.60,0.078706,0.904456,0.892,0.012456,0,54,0,446
5,Active Logistic Regression,sigmoid,0.983875,0.070861,0.37,0.081229,0.907373,0.892,0.015373,0,54,0,446


Value model metrics


,value_model,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,Random Forest Regressor,log1p(discounted_future_revenue_60d | active),0.882681,0.656536,0.514891,139.381051,0.886086,0.642423,0.496243,143.766239
1,Ridge Regression,log1p(discounted_future_revenue_60d | active),0.918765,0.686662,0.474418,166.320471,0.957195,0.713307,0.412145,191.129702
2,XGBoost Regressor,log1p(discounted_future_revenue_60d | active),0.919749,0.674425,0.473292,143.681740,0.912005,0.661173,0.466342,149.714869


## 6. Top-K ranking and profit scenario outputs

This is mainly useful as a **ranking layer**, not an auto-action engine.

### Top-K precision

`top_k_precision_summary.csv` shows whether the highest-risk customers have higher churn concentration than baseline. This is more relevant to M6 than global precision.

### Scenario profit

`scenario_profit_summary.csv` calculates expected incremental profit under business assumptions:

```text
expected_profit = p_churn_calibrated × save_rate × value_if_active × gross_margin - treatment_cost
```

If base scenario has zero profitable customers, the correct interpretation is:

> Do not auto-rollout paid voucher. Use high-risk customers as candidates for PSM / future campaign validation.

In [8]:
print('Top-K precision')
display(top_k_precision.head(20))

print('Scenario profit summary')
display(scenario_profit.head(20))

print('Profit threshold analysis')
display(profit_threshold.head(20))

Top-K precision


,score_name,top_k_share,top_k_customer_count,baseline_churn_rate,precision_at_k,lift_vs_baseline,churn_count_at_k,recall_at_k,min_score_in_top_k,mean_score_in_top_k,max_score_in_top_k
0,p_churn_calibrated,0.05,125,0.120848,0.568,4.700106,71,0.235099,0.276765,0.297303,0.356767
1,p_churn_calibrated,0.10,250,0.120848,0.448,3.707126,112,0.370861,0.235896,0.277311,0.356767
2,p_churn_calibrated,0.20,500,0.120848,0.330,2.730695,165,0.546358,0.155212,0.232298,0.356767
3,risk_value_score,0.05,125,0.120848,0.048,0.397192,6,0.019868,71.517129,90.462020,175.213680
4,risk_value_score,0.10,250,0.120848,0.064,0.529589,16,0.052980,56.175618,76.991626,175.213680
5,risk_value_score,0.20,500,0.120848,0.080,0.661987,40,0.132450,40.298270,61.991861,175.213680
6,expected_incremental_profit_base,0.05,125,0.120848,0.048,0.397192,6,0.019868,-4.106036,-3.869225,-2.809829
7,expected_incremental_profit_base,0.10,250,0.120848,0.064,0.529589,16,0.052980,-4.297805,-4.037605,-2.809829
8,expected_incremental_profit_base,0.20,500,0.120848,0.080,0.661987,40,0.132450,-4.496272,-4.225102,-2.809829


Scenario profit summary


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.0,0.0,750,-3657.133459
1,base,named,0.25,0.05,5.0,0,0.0,0.0,750,-3556.528039
2,optimistic,named,0.30,0.08,5.0,0,0.0,0.0,750,-3378.533835


Profit threshold analysis


,scenario,expected_profit_column,selection_rule,selected_customer_count,selected_customer_share,baseline_churn_rate,selected_churn_rate,lift_vs_baseline,total_expected_incremental_profit,mean_expected_incremental_profit,min_expected_incremental_profit,max_expected_incremental_profit,min_p_churn_calibrated,mean_p_churn_calibrated,max_p_churn_calibrated
0,conservative,expected_incremental_profit_conservative,profit_positive,0,0.00000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,expected_incremental_profit_conservative,top_5pct_by_expected_profit,125,0.05002,0.120848,0.048,0.397192,-557.153485,-4.457228,-4.570897,-3.948718,0.071547,0.097268,0.223377
2,conservative,expected_incremental_profit_conservative,top_10pct_by_expected_profit,250,0.10004,0.120848,0.064,0.529589,-1134.512562,-4.538050,-4.662946,-3.948718,0.068877,0.102500,0.327723
3,conservative,expected_incremental_profit_conservative,top_20pct_by_expected_profit,500,0.20008,0.120848,0.080,0.661987,-2314.024417,-4.628049,-4.758210,-3.948718,0.068848,0.106308,0.356420
4,base,expected_incremental_profit_base,profit_positive,0,0.00000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,base,expected_incremental_profit_base,top_5pct_by_expected_profit,125,0.05002,0.120848,0.048,0.397192,-483.653093,-3.869225,-4.106036,-2.809829,0.071547,0.097268,0.223377
6,base,expected_incremental_profit_base,top_10pct_by_expected_profit,250,0.10004,0.120848,0.064,0.529589,-1009.401170,-4.037605,-4.297805,-2.809829,0.068877,0.102500,0.327723
7,base,expected_incremental_profit_base,top_20pct_by_expected_profit,500,0.20008,0.120848,0.080,0.661987,-2112.550869,-4.225102,-4.496272,-2.809829,0.068848,0.106308,0.356420
8,optimistic,expected_incremental_profit_optimistic,profit_positive,0,0.00000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
9,optimistic,expected_incremental_profit_optimistic,top_5pct_by_expected_profit,125,0.05002,0.120848,0.048,0.397192,-353.613939,-2.828912,-3.283589,-0.794872,0.071547,0.097268,0.223377


## 7. Handoff files from the churn model

M5's main handoff files are in:

```text
models/m6_handoff/
```

Main files:

| File | Purpose |
|---|---|
| `high_risk_customers_for_ab_test.csv` | Top-risk candidate pool. Despite the old name, this can also support PSM subgroup analysis. |
| `priority_customers_all.csv` | Full customer ranking with risk/value/profit fields. |
| `churn_predictions.csv` | Full customer-level scoring output. |

These files are **not** proof that coupon treatment works. They only identify which households are high-risk or high-priority.

In [9]:
high_risk = read_m5_output('high_risk_customers_for_ab_test.csv', M6_HANDOFF_DIR)
priority_all = read_m5_output('priority_customers_all.csv', M6_HANDOFF_DIR)

print('High-risk customer file shape:', high_risk.shape)
display(show_selected(
    high_risk,
    [
        'household_key', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
        'risk_value_score', 'risk_value_rank', 'priority_segment',
        'predicted_discounted_value_60d_if_active',
        'expected_incremental_profit_base', 'recommended_treatment_action_base'
    ],
    n=10,
))

Loaded: models\m6_handoff\high_risk_customers_for_ab_test.csv
Loaded: models\m6_handoff\priority_customers_all.csv
High-risk customer file shape: (750, 65)


,household_key,p_churn_calibrated,risk_rank,risk_decile,risk_value_score,risk_value_rank,priority_segment,predicted_discounted_value_60d_if_active,expected_incremental_profit_base,recommended_treatment_action_base
0,2051,0.356767,1,1,26.579258,866,High Risk - Low Value,74.500429,-4.667759,A/B test candidate only; not profitable under ...
1,1974,0.356420,2,1,52.722586,283,High Risk - Low Value,147.922476,-4.340968,A/B test candidate only; not profitable under ...
2,2443,0.356420,3,1,36.118011,580,High Risk - Low Value,101.335426,-4.548525,A/B test candidate only; not profitable under ...
3,918,0.355589,4,1,22.094169,1035,High Risk - Low Value,62.133972,-4.723823,A/B test candidate only; not profitable under ...
4,370,0.354872,5,1,21.264586,1081,High Risk - Low Value,59.921891,-4.734193,A/B test candidate only; not profitable under ...
5,2092,0.354035,6,1,14.822133,1499,High Risk - Low Value,41.866307,-4.814723,A/B test candidate only; not profitable under ...
6,1031,0.352655,7,1,26.103431,888,High Risk - Low Value,74.019704,-4.673707,A/B test candidate only; not profitable under ...
7,2493,0.352655,8,1,19.784095,1191,High Risk - Low Value,56.100399,-4.752699,A/B test candidate only; not profitable under ...
8,93,0.347947,9,1,38.861523,536,High Risk - Low Value,111.688149,-4.514231,A/B test candidate only; not profitable under ...
9,2066,0.345082,10,1,8.683027,2020,High Risk - Low Value,25.162189,-4.891462,A/B test candidate only; not profitable under ...


# 8. PSM support for M6

M6 has chosen **Propensity Score Matching (PSM)** 

M5's role is limited and clear:

```text
Generate propensity_score = P(is_treated = 1 | observed pre-outcome covariates)
```

M6 will then use this score to perform matching, balance checks, and ROI/effect calculations.

**Important:** this file is an input for M5's PSM support script, not an M6 handoff file.

In [10]:
psm_flag_path = PSM_INPUTS_DIR / 'psm_treatment_flags.csv'
psm_template_path = PSM_INPUTS_DIR / 'psm_treatment_flags_TEMPLATE.csv'
psm_output_path = M6_HANDOFF_DIR / 'propensity_scores_for_psm.csv'

print('Expected PSM input from M4:', psm_flag_path.relative_to(PROJECT_ROOT))
print('PSM output for M6:', psm_output_path.relative_to(PROJECT_ROOT))

if psm_flag_path.exists():
    flags_preview = pd.read_csv(psm_flag_path)
    print('Treatment flag file found:', flags_preview.shape)
    display(flags_preview.head())
else:
    print('Treatment flag file not found yet.')
    print('Ask M4 to create:', psm_flag_path.relative_to(PROJECT_ROOT))
    if psm_template_path.exists():
        print('Template exists:', psm_template_path.relative_to(PROJECT_ROOT))

Expected PSM input from M4: models\psm_inputs\psm_treatment_flags.csv
PSM output for M6: models\m6_handoff\propensity_scores_for_psm.csv
Treatment flag file not found yet.
Ask M4 to create: models\psm_inputs\psm_treatment_flags.csv
Template exists: models\psm_inputs\psm_treatment_flags_TEMPLATE.csv


## 10. Optional dry-run only: create dummy treatment flags

Use this only if M4 has not delivered the real file and you want to test that the PSM script runs.

Do **not** use dummy propensity scores in the final report.

Set `CREATE_DUMMY_PSM_FLAGS = True` only for testing.

In [11]:
CREATE_DUMMY_PSM_FLAGS = False

if CREATE_DUMMY_PSM_FLAGS:
    import subprocess
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'create_dummy_psm_treatment_flags.py'),
        '--config', str(CONFIG_PATH),
        '--overwrite',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print('Skipped dummy PSM flag generation. Set CREATE_DUMMY_PSM_FLAGS=True for dry-run only.')

Skipped dummy PSM flag generation. Set CREATE_DUMMY_PSM_FLAGS=True for dry-run only.


## 9. Run the PSM propensity-score model

This step runs a simple Logistic Regression model to estimate:

```text
propensity_score = P(is_treated = 1 | covariates)
```

The script automatically excludes columns that are unsafe for PSM, such as:

```text
churn, future, active_flag, revenue_60d, p_churn, expected, profit, rank, decile, risk, coupon, promo, campaign, mailer, discount, redemption, recommendation, offer
```

This reduces leakage/collider-bias risk.

Set `RUN_PSM_PROPENSITY = True` only when `models/psm_inputs/psm_treatment_flags.csv` exists.

If using a dummy treatment file for dry-run, also set `ALLOW_DUMMY_PSM = True`. Do not report dummy output.

In [12]:
RUN_PSM_PROPENSITY = False
ALLOW_DUMMY_PSM = False

if RUN_PSM_PROPENSITY:
    import subprocess
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'psm_propensity_score.py'),
        '--config', str(CONFIG_PATH),
    ]
    if ALLOW_DUMMY_PSM:
        cmd.append('--allow-dummy')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print('Skipped PSM propensity score generation. Set RUN_PSM_PROPENSITY=True after M4 provides real treatment flags.')

Skipped PSM propensity score generation. Set RUN_PSM_PROPENSITY=True after M4 provides real treatment flags.


## 10. Review PSM output for M6

Final PSM handoff file:

```text
models/m6_handoff/propensity_scores_for_psm.csv
```

Minimum columns:

| Column | Meaning |
|---|---|
| `household_key` | Household identifier |
| `is_treated` | Treatment flag from M4 |
| `propensity_score` | Estimated probability of being treated |
| `propensity_logit` | Logit transform, useful for caliper matching |
| `common_support_flag` | Whether household falls inside treated/control common support |

M6 uses this file for matching and balance plots. M5 does not perform final matching or ROI calculation.

In [13]:
psm_scores = read_m5_output('propensity_scores_for_psm.csv', M6_HANDOFF_DIR)
if not psm_scores.empty:
    display(show_selected(
        psm_scores,
        [
            'household_key', 'is_treated', 'propensity_score', 'propensity_logit',
            'common_support_flag', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
            'priority_segment', 'predicted_discounted_value_60d_if_active'
        ],
        n=10,
    ))
else:
    print('No PSM score output yet. Run the PSM script after M4 provides real treatment flags.')

Missing: propensity_scores_for_psm.csv
No PSM score output yet. Run the PSM script after M4 provides real treatment flags.
